# App Feature A/B Test Analysis
## Nigerian Mobile-First Landing Page Experiment — Controlled Simulation

### Business question
> Does a lightweight, low-data landing page improve conversion and mobile user experience under Nigerian network conditions?

### Important methodology note
This is a **controlled experimental simulation**, not a live test of Nigerian customers. Nigerian telecom benchmark measurements are used as real contextual inputs; user-level assignments, performance outcomes and conversions are simulated. Results must therefore be interpreted only under the stated simulation assumptions.

## Assessment Alignment
- Working Jupyter deliverable
- Reproducible CSV dataset
- Data quality validation
- Primary KPI and hypothesis test
- 95% confidence interval
- Secondary KPI analysis
- Nigerian-context evidence
- Limitations and business recommendation


# 1. Import Libraries
The analysis uses pandas and NumPy for data work, matplotlib for visualization, SciPy/statsmodels for statistical analysis, and pathlib for file handling.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

print('Libraries imported successfully.')

# 2. Load the Simulation Dataset
The CSV contains simulated user-level observations for the Nigerian mobile-first experiment. It must be stored in the same directory as this notebook.

In [ ]:
data_path = Path('nigerian_mobile_ab_simulation.csv')
if not data_path.exists():
    raise FileNotFoundError('nigerian_mobile_ab_simulation.csv was not found. Place it beside this notebook.')

df = pd.read_csv(data_path, parse_dates=['timestamp'])
print(f'Rows: {df.shape[0]:,}')
print(f'Columns: {df.shape[1]}')
display(df.head())

# 3. Understand and Validate the Data
Before statistical testing, the dataset is checked for missing values, duplicate observations, duplicate users, expected experimental groups and valid binary outcomes.

In [ ]:
print('Missing values:')
display(df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate users:', df['user_id'].duplicated().sum())
print('Groups:')
display(df['experiment_group'].value_counts())
print('Conversion values:')
display(df['converted'].value_counts())

assert df['user_id'].is_unique
assert set(df['experiment_group'].unique()) == {'A_standard','B_lightweight'}
assert set(df['converted'].unique()).issubset({0,1})
assert set(df['bounced'].unique()).issubset({0,1})
assert set(df['cta_clicked'].unique()).issubset({0,1})
print('All validation checks passed.')

# 4. Nigerian Network Context
The simulation uses published Nigerian operator-level network benchmarks as contextual inputs. These are aggregate real-world measurements, not individual A/B observations.

The benchmark variables used are download speed, latency, Time to First Byte (TTFB), packet loss and operator sample distribution.

In [ ]:
telecom_benchmarks = pd.DataFrame({
    'operator':['Airtel','Glo','MTN'],
    'download_mbps':[21.74,12.85,37.63],
    'ttfb_ms':[1094,1027,892],
    'latency_ms':[163,152,151],
    'packet_loss_pct':[6.06,7.37,6.67],
    'sample_count':[61986,36011,192460]
})
telecom_benchmarks['sample_weight'] = telecom_benchmarks['sample_count'] / telecom_benchmarks['sample_count'].sum()
display(telecom_benchmarks)

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(telecom_benchmarks['operator'], telecom_benchmarks['download_mbps'])
plt.title('Published Nigerian Operator Download-Speed Benchmark')
plt.xlabel('Operator')
plt.ylabel('Download Speed (Mbps)')
plt.tight_layout()
plt.show()

# 5. Experimental Design
**Variant A — Standard:** conventional/larger landing-page payload.

**Variant B — Lightweight:** optimized low-data landing page.

**Primary KPI:** conversion rate.

**Secondary KPIs:** page-load time, data usage, bounce rate, CTA click-through rate and page size.

**H0:** pB = pA

**H1:** pB > pA

**Significance level:** α = 0.05

# 6. Simulation Methodology
The experiment contains 20,000 simulated users randomly assigned between the two variants. Operator assignment and network conditions reflect the published Nigerian benchmark context. Page performance and behavioral outcomes are generated using explicit modeling assumptions.

**Real inputs:** Nigerian telecom benchmark measurements.

**Simulated outputs:** users, experimental assignment, page performance, engagement and conversion outcomes.

In [ ]:
print('Experimental groups:')
display(df['experiment_group'].value_counts().sort_index())
print('Group proportions:')
display((df['experiment_group'].value_counts(normalize=True)*100).round(2))

# 7. Primary KPI — Conversion Rate
Conversion rate is calculated as conversions divided by users. The treatment effect is defined consistently as **B − A**.

In [ ]:
primary = df.groupby('experiment_group')['converted'].agg(conversions='sum', users='count', conversion_rate='mean')
display(primary)

rate_a = primary.loc['A_standard','conversion_rate']
rate_b = primary.loc['B_lightweight','conversion_rate']
absolute_uplift = rate_b - rate_a
relative_uplift = absolute_uplift / rate_a

print(f'A conversion rate: {rate_a:.2%}')
print(f'B conversion rate: {rate_b:.2%}')
print(f'Absolute uplift (B-A): {absolute_uplift:.2%}')
print(f'Relative uplift: {relative_uplift:.2%}')

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(['A — Standard','B — Lightweight'], [rate_a, rate_b])
plt.title('Conversion Rate by Experimental Variant')
plt.ylabel('Conversion Rate')
plt.ylim(0, max(rate_a, rate_b)*1.3)
plt.tight_layout()
plt.show()

# 8. Hypothesis Test
A two-proportion Z-test evaluates whether Variant B has a higher conversion rate than Variant A.

If p < 0.05, the null hypothesis is rejected. If p ≥ 0.05, there is insufficient evidence to reject it.

In [ ]:
z_stat, p_value = proportions_ztest(
    count=[primary.loc['B_lightweight','conversions'], primary.loc['A_standard','conversions']],
    nobs=[primary.loc['B_lightweight','users'], primary.loc['A_standard','users']],
    alternative='larger'
)
print(f'Z-statistic: {z_stat:.4f}')
print(f'P-value: {p_value:.6f}')
print('Decision:', 'Reject H0' if p_value < 0.05 else 'Fail to reject H0')

# 9. 95% Confidence Interval
The confidence interval is calculated for **B conversion rate − A conversion rate**.

If the interval is entirely above zero, it supports a positive treatment effect under the simulation assumptions. If it crosses zero, the treatment effect is inconclusive.

In [ ]:
ci_low, ci_high = confint_proportions_2indep(
    count1=primary.loc['B_lightweight','conversions'],
    nobs1=primary.loc['B_lightweight','users'],
    count2=primary.loc['A_standard','conversions'],
    nobs2=primary.loc['A_standard','users'],
    method='wald'
)
print(f'95% CI for B-A: [{ci_low:.4f}, {ci_high:.4f}]')
print(f'95% CI in percentage points: [{ci_low:.2%}, {ci_high:.2%}]')

# 10. Bootstrap Confidence Interval
As a robustness check, the conversion difference is resampled 5,000 times with replacement. The central 95% of the bootstrap distribution provides a non-parametric uncertainty interval.

In [ ]:
rng = np.random.default_rng(42)
a_values = df.loc[df['experiment_group']=='A_standard','converted'].to_numpy()
b_values = df.loc[df['experiment_group']=='B_lightweight','converted'].to_numpy()
bootstrap_effects = []

for _ in range(5000):
    a_sample = rng.choice(a_values, size=len(a_values), replace=True)
    b_sample = rng.choice(b_values, size=len(b_values), replace=True)
    bootstrap_effects.append(b_sample.mean() - a_sample.mean())

bootstrap_ci_low, bootstrap_ci_high = np.percentile(bootstrap_effects, [2.5,97.5])
print(f'Bootstrap 95% CI: [{bootstrap_ci_low:.2%}, {bootstrap_ci_high:.2%}]')

# 11. Secondary KPI Analysis
Conversion is the primary decision metric. The secondary metrics test whether the lightweight design also improves mobile performance and engagement.

In [ ]:
secondary = df.groupby('experiment_group').agg(
    average_page_load_ms=('page_load_ms','mean'),
    median_page_load_ms=('page_load_ms','median'),
    average_page_size_kb=('page_size_kb','mean'),
    average_data_usage_mb=('data_usage_mb','mean'),
    bounce_rate=('bounced','mean'),
    cta_click_rate=('cta_clicked','mean')
)
display(secondary.round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(['A — Standard','B — Lightweight'], secondary.loc[['A_standard','B_lightweight'],'average_page_load_ms'])
ax.set_title('Average Page-Load Time')
ax.set_ylabel('Milliseconds')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(['A — Standard','B — Lightweight'], secondary.loc[['A_standard','B_lightweight'],'average_data_usage_mb'])
ax.set_title('Average Data Usage')
ax.set_ylabel('MB')
plt.tight_layout()
plt.show()

In [ ]:
x=np.arange(2); width=0.35
bounce=secondary.loc[['A_standard','B_lightweight'],'bounce_rate'].to_numpy()
cta=secondary.loc[['A_standard','B_lightweight'],'cta_click_rate'].to_numpy()
fig, ax=plt.subplots(figsize=(9,5))
ax.bar(x-width/2,bounce,width,label='Bounce rate')
ax.bar(x+width/2,cta,width,label='CTA click rate')
ax.set_xticks(x); ax.set_xticklabels(['A — Standard','B — Lightweight'])
ax.set_ylabel('Rate'); ax.set_title('Engagement Metrics by Variant'); ax.legend()
plt.tight_layout(); plt.show()

# 12. Operator-Level Exploratory Analysis
The operator breakdown is descriptive only. It was not separately powered for operator-level causal inference.

In [ ]:
operator_results = df.groupby(['operator','experiment_group'])['converted'].agg(conversions='sum',users='count',conversion_rate='mean').reset_index()
display(operator_results.round(4))
operator_pivot = operator_results.pivot(index='operator',columns='experiment_group',values='conversion_rate')
operator_pivot.plot(kind='bar', figsize=(9,5))
plt.title('Simulated Conversion Rate by Operator')
plt.ylabel('Conversion Rate')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# 13. Sensitivity Analysis
Because this is a simulation, assumptions matter. We compare hypothetical lightweight page sizes as scenarios. These are not additional observed results.

In [ ]:
scenario_results=[]
for page_kb in [500,750,1000,1250]:
    mask=df['experiment_group']=='B_lightweight'
    adjusted_load=df.loc[mask,'page_load_ms']*(page_kb/df.loc[mask,'page_size_kb'].mean())
    scenario_results.append({'B_page_size_kb':page_kb,'average_adjusted_load_ms':adjusted_load.mean()})
sensitivity=pd.DataFrame(scenario_results)
display(sensitivity)

# 14. Final Scorecard and Business Recommendation
The primary decision is based on conversion rate, treatment effect and statistical significance. Secondary metrics provide user-experience guardrails.

In [ ]:
scorecard=pd.DataFrame({
    'Metric':['Conversion rate','Absolute uplift','Relative uplift','P-value','95% CI lower','95% CI upper','Avg page-load ms','Avg data usage MB','Bounce rate','CTA click rate'],
    'A_Standard':[rate_a,np.nan,np.nan,np.nan,np.nan,np.nan,secondary.loc['A_standard','average_page_load_ms'],secondary.loc['A_standard','average_data_usage_mb'],secondary.loc['A_standard','bounce_rate'],secondary.loc['A_standard','cta_click_rate']],
    'B_Lightweight':[rate_b,absolute_uplift,relative_uplift,p_value,ci_low,ci_high,secondary.loc['B_lightweight','average_page_load_ms'],secondary.loc['B_lightweight','average_data_usage_mb'],secondary.loc['B_lightweight','bounce_rate'],secondary.loc['B_lightweight','cta_click_rate']]
})
display(scorecard)

if p_value < 0.05 and absolute_uplift > 0:
    print('Recommendation: advance Variant B to a live validation experiment.')
else:
    print('Recommendation: do not advance Variant B without further testing.')
print('Important: this recommendation is based on simulated observations, not real production users.')

# 15. Limitations and Next Step
### Limitations
1. Individual user behavior is simulated.
2. Conversion outcomes depend on model assumptions.
3. Nigerian network inputs are aggregate benchmark measurements.
4. There was no live production exposure.
5. Results cannot be generalized to all Nigerian users.

### Production next step
Run a live randomized experiment using the same primary KPI, secondary guardrails and statistical framework. Capture real device, operator, network, performance, engagement and conversion data.

# Final Portfolio Conclusion
This project demonstrates an end-to-end A/B testing workflow: business problem definition → experimental design → data validation → Nigerian market context → conversion analysis → hypothesis testing → confidence intervals → secondary KPI analysis → sensitivity analysis → business recommendation.

The key analytical discipline is to clearly distinguish **real contextual evidence** from **simulated user-level outcomes**.